# Lab 0 — A Sentinel-2 scene of Taipei: open it, decode it, look at it, measure it
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trongan93/dl-space-2026/blob/main/notebooks/Week2_Lab0_Sentinel2_GeoTIFF.ipynb)  ·  repo: [trongan93/dl-space-2026](https://github.com/trongan93/dl-space-2026)

**Deep Learning in Space Technology Applications · 115-1 · Week 2 (14 Sep 2026) · due Sunday 20 Sep 2026, 23:59 on i-School Plus**

| | |
|---|---|
| Name / student ID | *fill in* |
| Course code | 366257 / 366646 |
| Scene ID used | *filled by the notebook in Part 1* |

### What you will do
1. Search a catalogue (STAC) for a cloud-free Sentinel-2 L2A scene over Taipei — no download of the whole tile.
2. Read six bands plus the Scene Classification Layer for one 10 km window, straight from cloud-optimised GeoTIFFs.
3. Decode digital numbers to surface reflectance, honour *nodata*, apply the cloud mask.
4. Make a true-colour and two false-colour composites (for **you**), and keep the calibrated cube (for the **model**).
5. Compute NDVI and NDWI, look at their histograms, estimate the water fraction of the window.
6. Answer five short questions. **The code is given; the marks are for the answers and for correct decoding.**

### Rules
* Run every cell top to bottom (*Runtime → Run all*) before exporting: *File → Print → Save as PDF* (or *File → Download → .ipynb* plus a PDF).
* Cells marked **✏️ YOUR ANSWER** must be filled in. Everything else may be left as is.
* Runs on Google Colab free tier in about ten minutes; no account is needed for the data.
* Optional stretch (worth doing for your team): change `BBOX` in the configuration cell to the area of your HW1 problem.

## 0 · Setup
Colab does not ship `rasterio` or `pystac-client`. The next cell installs them (about one minute). If you run locally, `pip install rasterio pystac-client matplotlib numpy` once.

In [ ]:
import importlib, subprocess, sys
for pkg, mod in [("rasterio", "rasterio"), ("pystac-client", "pystac_client")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import os, json, math, datetime as dt
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
from rasterio.transform import from_origin
print("rasterio", rasterio.__version__, "| numpy", np.__version__)

## 1 · Configuration — the search contract
Everything about *which pixels* you will get is decided here, before a single byte is read. This cell is the seed of the data contract you will write in Lab 1.

* `BBOX` is in longitude/latitude (WGS84, EPSG:4326) because the catalogue is searched in geographic coordinates.
* The window is read on the scene's own UTM grid (Taiwan: zone 51N, **EPSG:32651**) so that pixels stay square 10 m cells.
* Bands are named with the catalogue's asset keys; the Sentinel-2 band number is given alongside.

In [ ]:
# --- area of interest: Tamsui river mouth to the Taipei basin (about 27 km x 22 km) ---
BBOX = [121.35, 25.00, 121.65, 25.20]          # [west, south, east, north] in degrees
DATE_RANGE = "2026-01-01/2026-09-10"
MAX_CLOUD = 20                                 # percent, scene-level metadata
GSD = 10                                       # metres: everything is resampled to this grid
MAX_WINDOW_PX = 1200                           # safety cap so Colab RAM is not exhausted

# Earth Search asset key  ->  (Sentinel-2 band, native GSD, role)
BANDS = {
    "blue":   ("B02", 10, "blue"),
    "green":  ("B03", 10, "green"),
    "red":    ("B04", 10, "red"),
    "nir":    ("B08", 10, "NIR"),
    "swir16": ("B11", 20, "SWIR-1"),
    "swir22": ("B12", 20, "SWIR-2"),
}
BAND_ORDER = list(BANDS)                       # index into the cube: 0 blue ... 5 swir22
SCL_VALID = [4, 5, 6, 7, 11]                   # vegetation, not-vegetated, water, unclassified, snow/ice
L2A_OFFSET, L2A_SCALE = 1000, 10000            # processing baseline >= 04.00 (Jan 2022)

STAC_URL = "https://earth-search.aws.element84.com/v1"
COLLECTION = "sentinel-2-l2a"
FORCE_FALLBACK = bool(os.environ.get("LAB0_FORCE_FALLBACK"))   # instructor testing switch
REPO_RAW = "https://raw.githubusercontent.com/trongan93/dl-space-2026/main/data/"   # course sample scene (real Sentinel-2, bundled in the repo) - tier 2 fallback
SAMPLE_TIF, SAMPLE_SCL = "taipei_s2_sample.tif", "taipei_s2_sample_scl.tif"
print("Search:", BBOX, DATE_RANGE, f"cloud < {MAX_CLOUD}%")

## 2 · Search the catalogue (STAC)
A STAC catalogue is a JSON index of scenes: footprint, acquisition time, cloud cover, and one HTTPS link per band file. We search first and read only the window we need.

If the classroom network blocks AWS the cell falls back to a **clearly labelled synthetic scene** so the rest of the notebook still runs. Synthetic results are *not* acceptable for submission — rerun at home.

In [ ]:
def stac_search():
    from pystac_client import Client
    cat = Client.open(STAC_URL)
    search = cat.search(collections=[COLLECTION], bbox=BBOX, datetime=DATE_RANGE,
                        query={"eo:cloud_cover": {"lt": MAX_CLOUD}}, max_items=50)
    items = list(search.item_collection())
    if not items:
        raise RuntimeError("No scenes matched - relax MAX_CLOUD or widen DATE_RANGE")
    items.sort(key=lambda i: i.properties["eo:cloud_cover"])
    return items

SYNTHETIC = False; SAMPLE = False
try:
    if FORCE_FALLBACK:
        raise RuntimeError("fallback forced")
    items = stac_search()
    print(f"{len(items)} scenes with cloud < {MAX_CLOUD}%  (sorted, least cloudy first)")
    for it in items[:8]:
        p = it.properties
        print(f"  {it.id:40s} {p['datetime'][:10]}  cloud {p['eo:cloud_cover']:5.1f}%  tile {p.get('grid:code', p.get('s2:mgrs_tile', '?'))}")
    item = items[0]
    SCENE_ID = item.id
except Exception as e:
    item = None
    print("Catalogue not reachable:", repr(e)[:120])
    try:   # tier 2: the course's bundled real scene
        import urllib.request
        for fn in (SAMPLE_TIF, SAMPLE_SCL):
            if not os.path.exists(fn): urllib.request.urlretrieve(REPO_RAW + fn, fn)
        SAMPLE = True
        with rasterio.open(SAMPLE_TIF) as src: SCENE_ID = src.tags().get("scene_id", "bundled sample") + " (course sample - say so in your answers)"
        print(">>> Using the course's bundled Sentinel-2 sample scene (real data, not your own search).")
    except Exception as e2:
        SYNTHETIC = True
        SCENE_ID = "SYNTHETIC-SCENE (network fallback) - NOT VALID FOR SUBMISSION"
        print("Sample not reachable either:", repr(e2)[:100])
        print(">>> Falling back to a synthetic scene so the notebook can run. Rerun with network for real data.")
print("\nScene used:", SCENE_ID)

## 3 · Read one window, six bands, on one 10 m grid
Three details carry most of the marks in this lab:

1. **Window, not tile.** A Sentinel-2 tile is 10 980 × 10 980 pixels per 10 m band — too big for Colab. `rasterio` reads only the window that covers the bounding box, over HTTPS, from a cloud-optimised GeoTIFF.
2. **Two resampling rules.** Reflectance bands (20 m → 10 m) use *bilinear*; the SCL class map uses *nearest* — averaging class numbers would invent classes.
3. **Decode.** Level-2A stores `DN = reflectance * 10000 + 1000`; `DN = 0` means *no data*.

In [ ]:
def read_window(href, bounds_utm, out_shape, resampling):
    # Read one band for `bounds_utm` (in the file's CRS) resampled to `out_shape`.
    with rasterio.open(href) as src:
        win = from_bounds(*bounds_utm, transform=src.transform)
        arr = src.read(1, window=win, out_shape=out_shape, resampling=resampling, boundless=True, fill_value=0)
        return arr, src.crs, rasterio.windows.transform(win, src.transform)

def decode_l2a(dn):
    # uint16 DN -> float32 surface reflectance, NaN where nodata
    rho = (dn.astype("float32") - L2A_OFFSET) / L2A_SCALE
    rho[dn == 0] = np.nan
    return rho

if SAMPLE:
    with rasterio.open(SAMPLE_TIF) as src:
        cube, cube_crs, cube_transform = src.read().astype("float32"), src.crs, src.transform
        H, W = cube.shape[1:]; bounds_utm = tuple(src.bounds)
    with rasterio.open(SAMPLE_SCL) as src: scl = src.read(1).astype("uint8")
    print("sample cube read:", cube.shape, "| CRS", cube_crs)
elif not SYNTHETIC:
    # 1. the scene's CRS (UTM) and the bbox in that CRS
    with rasterio.open(item.assets["red"].href) as src:
        scene_crs = src.crs
        native_transform = src.transform
    bounds_utm = transform_bounds("EPSG:4326", scene_crs, *BBOX)
    W = min(int(round((bounds_utm[2] - bounds_utm[0]) / GSD)), MAX_WINDOW_PX)
    H = min(int(round((bounds_utm[3] - bounds_utm[1]) / GSD)), MAX_WINDOW_PX)
    # trim bounds to the capped size, anchored at the north-west corner
    bounds_utm = (bounds_utm[0], bounds_utm[3] - H * GSD, bounds_utm[0] + W * GSD, bounds_utm[3])
    print("scene CRS:", scene_crs, "| window", H, "rows x", W, "cols at", GSD, "m")

    # 2. six reflectance bands
    layers = []
    for key in BAND_ORDER:
        dn, _, win_transform = read_window(item.assets[key].href, bounds_utm, (H, W), Resampling.bilinear)
        layers.append(decode_l2a(dn))
        print(f"  {key:7s} {BANDS[key][0]}  native {BANDS[key][1]:2d} m  DN range {dn.min():5d}-{dn.max():5d}")
    cube = np.stack(layers)                                   # (6, H, W) float32
    # 3. the class map - nearest neighbour!
    scl, _, _ = read_window(item.assets["scl"].href, bounds_utm, (H, W), Resampling.nearest)
    cube_transform = from_origin(bounds_utm[0], bounds_utm[3], GSD, GSD)
    cube_crs = scene_crs
else:
    # ---------- synthetic fallback: a plausible-looking 6-band scene, NOT real data ----------
    rng = np.random.default_rng(0)
    H, W = 600, 800
    yy, xx = np.mgrid[0:H, 0:W] / 100.0
    river = np.abs(yy - 3 - 1.2 * np.sin(xx / 1.5)) < 0.25
    sea = xx < 1.2 + 0.3 * np.sin(yy)
    urban = (xx > 3.5) & (yy > 3.5) & ~river
    water = river | sea
    veg = ~water & ~urban
    def band(w_val, v_val, u_val, noise=0.02):
        a = np.where(water, w_val, np.where(urban, u_val, v_val)).astype("float32")
        return np.clip(a + rng.normal(0, noise, a.shape), 0.001, 0.95)
    cube = np.stack([band(0.06, 0.04, 0.12), band(0.08, 0.07, 0.14), band(0.05, 0.05, 0.16),
                     band(0.02, 0.45, 0.22), band(0.01, 0.25, 0.30), band(0.005, 0.15, 0.28)]).astype("float32")
    cloud = ((xx - 6) ** 2 + (yy - 1.5) ** 2) < 0.6
    cube[:, cloud] = 0.7
    scl = np.where(water, 6, np.where(urban, 5, 4)).astype("uint8"); scl[cloud] = 9
    cube[:, :15, :] = np.nan                                  # a nodata strip, as at a tile edge
    scl[:15, :] = 0
    cube_crs = rasterio.crs.CRS.from_epsg(32651)
    cube_transform = from_origin(290000, 2790000, GSD, GSD)
    print("SYNTHETIC cube generated:", cube.shape)

valid = np.isin(scl, SCL_VALID) & np.isfinite(cube).all(axis=0)
print("\ncube shape (C, H, W):", cube.shape, cube.dtype)
print("reflectance range (valid pixels): %.3f - %.3f" % (np.nanmin(cube[:, valid]), np.nanmax(cube[:, valid])))
print("valid fraction after SCL + nodata mask: %.1f %%" % (100 * valid.mean()))

## 4 · The GeoTIFF contract — write it, read it back
A GeoTIFF is the array **plus** four things: coordinate reference system, affine transform, nodata value and tags. Writing your cube out and reading it back proves you carried all four.

In [ ]:
out_tif = "lab0_cube.tif"
profile = dict(driver="GTiff", dtype="float32", count=cube.shape[0], height=cube.shape[1], width=cube.shape[2],
               crs=cube_crs, transform=cube_transform, nodata=np.nan, compress="deflate", tiled=True)
with rasterio.open(out_tif, "w", **profile) as dst:
    dst.write(np.nan_to_num(cube, nan=np.nan))
    dst.descriptions = tuple(f"{k} {BANDS[k][0]} {BANDS[k][2]}" for k in BAND_ORDER)
    dst.update_tags(scene_id=SCENE_ID, decode="rho=(DN-1000)/10000", scl_valid=str(SCL_VALID),
                    course="Deep Learning in Space Technology Applications Lab 0")
np.save("lab0_scl.npy", scl)

with rasterio.open(out_tif) as src:
    print("count / height / width / dtype :", src.count, src.height, src.width, src.dtypes[0])
    print("CRS                            :", src.crs)
    print("transform (a, b, c, d, e, f)   :", tuple(round(v, 2) for v in src.transform[:6]))
    print("pixel size (res)               :", src.res)
    print("nodata                         :", src.nodata)
    print("band descriptions              :", src.descriptions)
    print("tags                           :", src.tags())
    e, n = src.xy(0, 0)
    print("centre of pixel (row 0, col 0) :", f"E {e:,.0f} m  N {n:,.0f} m  in {src.crs}")
    print("file size                      : %.1f MB" % (os.path.getsize(out_tif) / 1e6))

## 5 · Look at it — composites are for humans
Reflectance lives in 0–1 with most land below 0.3, so a straight display is nearly black. We stretch each channel between its 2nd and 98th percentile **for display only**. The model never sees the stretched image.

* **True colour** RGB = (B4, B3, B2)
* **Colour infrared** RGB = (B8, B4, B3): vegetation bright red, water black
* **SWIR composite** RGB = (B12, B8, B4): urban and bare soil bright, burn scars red-brown

In [ ]:
def stretch(img, lo=2, hi=98):
    # Per-channel percentile stretch to [0,1] for display. img: (3, H, W), NaNs allowed
    out = np.zeros_like(img, dtype="float32")
    for i in range(img.shape[0]):
        band = img[i]
        a, b = np.nanpercentile(band[valid], [lo, hi])
        out[i] = np.clip((band - a) / (b - a + 1e-9), 0, 1)
    return np.nan_to_num(np.moveaxis(out, 0, -1))          # (H, W, 3)

I = {k: i for i, k in enumerate(BAND_ORDER)}
composites = {
    "True colour  RGB=(B4,B3,B2)":        cube[[I["red"], I["green"], I["blue"]]],
    "Colour infrared  RGB=(B8,B4,B3)":    cube[[I["nir"], I["red"], I["green"]]],
    "SWIR composite  RGB=(B12,B8,B4)":    cube[[I["swir22"], I["nir"], I["red"]]],
}
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, arr) in zip(axes, composites.items()):
    ax.imshow(stretch(arr)); ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"{SCENE_ID}  |  window {cube.shape[2]} x {cube.shape[1]} px at {GSD} m", y=0.98)
plt.tight_layout(); plt.show()

### 5b · The class map the product came with
SCL is produced by an algorithm (Sen2Cor), not by a person. Over Taipei it usually mislabels some bright roofs and misses thin cloud over water. Compare it with the true-colour image.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
scl_names = ["0 no data", "1 saturated", "2 dark", "3 cloud shadow", "4 vegetation", "5 not vegetated",
             "6 water", "7 unclassified", "8 cloud med", "9 cloud high", "10 thin cirrus", "11 snow/ice"]
scl_colors = ["#000000", "#ff0000", "#404040", "#833c0b", "#00a000", "#ffff00",
              "#0000cc", "#808080", "#c0c0c0", "#ffffff", "#66ccff", "#ff66ff"]
cmap = ListedColormap(scl_colors); norm = BoundaryNorm(np.arange(-0.5, 12.5, 1), cmap.N)
fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(scl, cmap=cmap, norm=norm); ax.set_xticks([]); ax.set_yticks([])
cb = plt.colorbar(im, ax=ax, ticks=range(12), fraction=0.03); cb.ax.set_yticklabels(scl_names)
ax.set_title("Scene Classification Layer (SCL)"); plt.tight_layout(); plt.show()

counts = np.bincount(scl.ravel(), minlength=12) / scl.size * 100
for k, (nm, pct) in enumerate(zip(scl_names, counts)):
    if pct > 0.05: print(f"  {nm:16s} {pct:5.1f} %  {'(valid)' if k in SCL_VALID else ''}")

## 6 · Measure it — indices and histograms
$$\mathrm{NDVI} = \frac{B8 - B4}{B8 + B4}, \qquad \mathrm{NDWI} = \frac{B3 - B8}{B3 + B8}$$

Normalised differences cancel illumination and expose a material contrast. They are also your first sanity check: if you forgot the L2A offset, NDVI of dense vegetation comes out too low and water no longer goes negative.

In [ ]:
def nd(a, b):
    with np.errstate(invalid="ignore", divide="ignore"):
        return (a - b) / (a + b)

ndvi = nd(cube[I["nir"]], cube[I["red"]])
ndwi = nd(cube[I["green"]], cube[I["nir"]])
ndvi_m = np.where(valid, ndvi, np.nan); ndwi_m = np.where(valid, ndwi, np.nan)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
im0 = axes[0, 0].imshow(ndvi_m, cmap="RdYlGn", vmin=-0.2, vmax=0.9); axes[0, 0].set_title("NDVI (masked)"); plt.colorbar(im0, ax=axes[0, 0], fraction=0.03)
im1 = axes[0, 1].imshow(ndwi_m, cmap="Blues", vmin=-0.6, vmax=0.6); axes[0, 1].set_title("NDWI (masked)"); plt.colorbar(im1, ax=axes[0, 1], fraction=0.03)
axes[1, 0].hist(ndvi_m[valid], bins=100, color="#2D7D5B"); axes[1, 0].set_title("NDVI histogram, valid pixels"); axes[1, 0].set_xlim(-1, 1)
axes[1, 1].hist(ndwi_m[valid], bins=100, color="#3066BE"); axes[1, 1].set_title("NDWI histogram, valid pixels"); axes[1, 1].set_xlim(-1, 1)
for ax in axes[0]: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

water_ndwi = (ndwi_m > 0)
water_scl = (scl == 6) & valid
print("water fraction of valid pixels - NDWI > 0 : %.1f %%" % (100 * water_ndwi[valid].mean()))
print("water fraction of valid pixels - SCL == 6 : %.1f %%" % (100 * water_scl[valid].mean()))
agree = (water_ndwi == water_scl)[valid].mean()
print("pixel agreement between the two water masks: %.1f %%" % (100 * agree))
print("NDVI of the greenest 1 %% of pixels: %.2f  |  NDVI of SCL-water pixels (median): %.2f"
      % (np.nanpercentile(ndvi_m[valid], 99), np.nanmedian(ndvi_m[water_scl]) if water_scl.any() else float('nan')))

## 7 · Raw digital numbers — see the offset with your own eyes
This cell re-reads the red band **without decoding** and plots the histogram of the stored integers. Look for the spike at 0 (*nodata*) and where the land values start (~1000 + reflectance × 10000).

In [ ]:
if item is not None:
    dn_red, _, _ = read_window(item.assets["red"].href, bounds_utm, (cube.shape[1], cube.shape[2]), Resampling.nearest)
else:
    dn_red = np.nan_to_num(cube[I["red"]] * L2A_SCALE + L2A_OFFSET, nan=0).astype("uint16")
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.hist(dn_red.ravel(), bins=200, range=(0, 6000), color="#B05A2B")
ax.axvline(L2A_OFFSET, color="k", ls="--"); ax.text(L2A_OFFSET + 40, ax.get_ylim()[1] * 0.85, "DN = 1000  <=>  reflectance 0")
ax.set_xlabel("stored uint16 digital number, band B04 (red)"); ax.set_ylabel("pixels"); ax.set_yscale("log")
ax.set_title("Raw DN histogram - the offset and the nodata spike"); plt.tight_layout(); plt.show()
print("dtype in file:", dn_red.dtype, "| min", dn_red.min(), "| max", dn_red.max(), "| nodata pixels:", int((dn_red == 0).sum()))

## 8 · ✏️ YOUR ANSWERS (marked)
Write two to four sentences each, in English, in the cell below. Refer to numbers printed above.

1. **Decoding.** What is the stored integer range of the red band in your window, and what reflectance range does it decode to? Which formula did you apply, and what would NDVI of dense vegetation have looked like if you had forgotten the −1000 offset?
2. **Mixed GSD.** Two of your six bands were resampled from 20 m to 10 m. Which two, which resampling did you use for them, and why was a *different* resampling used for the SCL layer?
3. **Masks.** What fraction of the window did the SCL + nodata mask remove? Find one place where SCL looks wrong compared with the true-colour image and describe it (row/col or a landmark).
4. **Water.** Compare the NDWI > 0 water fraction with the SCL water fraction. Which do you trust more here, and what threshold (if not 0) would you use for the Tamsui river?
5. **Your HW1 target.** Using this scene's GSD (10 m), how many pixels across would your HW1 target be? Is Sentinel-2 the right sensor for it? If not, which one, and what does that change about revisit and access?

✏️ **YOUR ANSWER**

1.

2.

3.

4.

5.

## 9 · Optional stretch — your own area, and a file for Lab 1
Change `BBOX` in Part 1 to the area from your HW1 problem statement and rerun. Then run the cell below to save the cube and mask as NumPy arrays; Lab 1 (dataset contract) starts from these files.

In [ ]:
np.save("lab0_cube.npy", cube.astype("float32"))
np.save("lab0_valid.npy", valid)
meta = dict(scene_id=SCENE_ID, synthetic=SYNTHETIC, course_sample=SAMPLE, bbox_wgs84=BBOX, crs=str(cube_crs), gsd_m=GSD,
            transform=list(cube_transform)[:6], bands=[f"{k}:{BANDS[k][0]}" for k in BAND_ORDER],
            decode="rho=(DN-1000)/10000", scl_valid=SCL_VALID, shape=list(cube.shape),
            created=dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"))
json.dump(meta, open("lab0_meta.json", "w"), indent=2)
print(json.dumps(meta, indent=2))
print("\nSaved: lab0_cube.tif, lab0_cube.npy, lab0_valid.npy, lab0_scl.npy, lab0_meta.json")

---
### Checklist before you export
- [ ] All cells executed, no error output
- [ ] Scene ID is a real Sentinel-2 product (not the synthetic fallback)
- [ ] Three composites, SCL map, NDVI/NDWI maps and histograms, raw-DN histogram are visible
- [ ] Five answers written
- [ ] Name and student ID at the top

*Data: Copernicus Sentinel-2 Level-2A, via the AWS Open Data Sentinel-2 COG archive and the Element 84 Earth Search STAC API. Contains modified Copernicus Sentinel data 2026.*